# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Engagement rate collapse in decaying pages
The Paper Finding: The FlyRank Research Paper identifies that pages experiencing traffic drops exhibit an engagement rate drop of ~50% compared to growing pages, claiming engagement deterioration precedes ranking losses.


Constructive Methodology Questions:


Label origin & Definition: How was engagement rate measured across distinct site structures? In raw tabular slices, engagement rate is frequently zero-inflated (median 0.00% across thousands of rows), indicating either instrumentation missingness or tracking gaps rather than genuine user disinterest.


Confounding & Causality: Does lower engagement cause ranking loss, or does losing top-tier ranking cause search engines to route lower-intent or misaligned traffic to the URL? A correlation between drop direction and engagement does not establish temporal precedence without a longitudinal cohort test.


Finding 2: AI Overviews displacing organic search CTR
The Paper Finding: The paper notes an accelerated drop in click-through rates (CTR) on informational queries following the broader deployment of Google AI Overviews.


Constructive Methodology Questions:


SERP Feature Verification: Was the presence of an AI Overview directly observed via contemporaneous SERP scrapers for each specific query, or was it inferred post-hoc from aggregate CTR declines?


Selection Bias: High-impression informational head queries are naturally more susceptible to layout changes than transactional or long-tail keywords. Without controlling for query intent classification, attributing the CTR reduction specifically to AI Overviews risks conflating routine seasonality and search intent shifts with algorithmic layout changes.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Split Design Comparison
Week-5 Baseline (Random Stratified Split): An 80/20 stratified split on is_high_value_decay treats every page independently. This allows pages sharing identical catalog age cohorts or topical clusters to appear simultaneously in both train and test splits.

Week-6 Honest Split (Age-Cohort / Time-Aware Split): Pages are sorted chronologically by content_age_days. The older 80% of assets serve as the training set, and the newest 20% serve as the test holdout. This simulates the real production setting: training on existing historical pages to score newly aging incoming pages.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# 1. Load Data & Prepare Signals
df = pd.read_excel("capstone_data.xlsx")
df["trend_pct_num"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
).fillna(0)
df["clicks_90d"] = pd.to_numeric(df["clicks_90d"], errors="coerce").fillna(0)
df["content_age_days"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
).fillna(0)
df["days_since_last_update"] = pd.to_numeric(
    df.get("days_since_last_update", 0), errors="coerce"
).fillna(0)
df["feat_ctr_90d"] = np.where(
    df["impressions_90d"] > 0, df["clicks_90d"] / df["impressions_90d"], 0.0
)

# Label Definition
df["is_high_value_decay"] = (
    (df["trend_direction"] == "down")
    & (df["impressions_90d"] >= 1000)
    & (df["trend_pct_num"] <= -20)
).astype(int)

features = [
    "impressions_90d",
    "feat_ctr_90d",
    "content_age_days",
    "days_since_last_update",
]
X = df[features]
y = df["is_high_value_decay"]

# ---------------------------------------------------------
# SETUP A: Week-5 Stratified Random Split
# ---------------------------------------------------------
X_tr_rnd, X_te_rnd, y_tr_rnd, y_te_rnd, _, idx_te_rnd = train_test_split(
    X, y, df.index, test_size=0.20, random_state=42, stratify=y
)
rf_rnd = RandomForestClassifier(
    n_estimators=150, max_depth=6, min_samples_leaf=5, random_state=42
)
rf_rnd.fit(X_tr_rnd, y_tr_rnd)
prob_rnd = rf_rnd.predict_proba(X_te_rnd)[:, 1]

df_rnd_test = df.loc[idx_te_rnd].copy()
df_rnd_test["prob"] = prob_rnd
top50_rnd = df_rnd_test.sort_values(by="prob", ascending=False).head(50)
p50_rnd = top50_rnd["is_high_value_decay"].mean()
auc_rnd = roc_auc_score(y_te_rnd, prob_rnd)

# ---------------------------------------------------------
# SETUP B: Week-6 Honest Time/Age-Cohort Split
# Train on older 80% of assets, test on newer 20%
# ---------------------------------------------------------
df_sorted = df.sort_values(by="content_age_days", ascending=False).reset_index(
    drop=True
)
split_idx = int(len(df_sorted) * 0.80)

train_cohort = df_sorted.iloc[:split_idx]
test_cohort = df_sorted.iloc[split_idx:].copy()

rf_cohort = RandomForestClassifier(
    n_estimators=150, max_depth=6, min_samples_leaf=5, random_state=42
)
rf_cohort.fit(train_cohort[features], train_cohort["is_high_value_decay"])

test_cohort["prob"] = rf_cohort.predict_proba(test_cohort[features])[:, 1]
top50_cohort = test_cohort.sort_values(by="prob", ascending=False).head(50)
p50_cohort = top50_cohort["is_high_value_decay"].mean()
auc_cohort = roc_auc_score(
    test_cohort["is_high_value_decay"], test_cohort["prob"]
)

# Before vs. After Summary Table
audit_table = pd.DataFrame(
    [
        {
            "Split Strategy": "Random Stratified (Week 5)",
            "Precision@50": f"{p50_rnd:.2%}",
            "ROC-AUC": f"{auc_rnd:.4f}",
            "Honesty Rationale": "Baseline validation; shares age distribution across splits",
        },
        {
            "Split Strategy": "Age-Cohort Split (Week 6 Audit)",
            "Precision@50": f"{p50_cohort:.2%}",
            "ROC-AUC": f"{auc_cohort:.4f}",
            "Honesty Rationale": "Stricter temporal simulation; tests generalization on newer assets",
        },
    ]
)

display(audit_table)

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Pipeline InspectionFuture-Window Leakage: Verified that post-decision metrics—such as post_refresh_clicks_30d—remain completely excluded from feature sets across all notebooks.Target Proxy Verification: In w03_data_contract.ipynb, we tested an intentional target leakage trap where target_is_high_value_decay leaked into the feature space, yielding an artificial 1.0000 ROC-AUC. In our audited production pipeline, trend_pct and trend_direction are excluded from the model feature matrix $X$ (impressions_90d, feat_ctr_90d, content_age_days, days_since_last_update), eliminating direct proxy circularity.

In [ ]:
# Extract and display real error examples
test_cohort["rank"] = test_cohort["prob"].rank(ascending=False)
fp_audit = test_cohort[
    (test_cohort["rank"] <= 50) & (test_cohort["is_high_value_decay"] == 0)
]
fn_audit = test_cohort[
    (test_cohort["rank"] > 50) & (test_cohort["is_high_value_decay"] == 1)
]

print(f"False Positives in Top 50: {len(fp_audit)}")
print(
    fp_audit[
        [
            "content_id",
            "impressions_90d",
            "trend_pct_num",
            "content_age_days",
            "prob",
        ]
    ].head(3)
)

print(f"\nFalse Negatives outside Top 50: {len(fn_audit)}")
print(
    fn_audit[
        [
            "content_id",
            "impressions_90d",
            "trend_pct_num",
            "content_age_days",
            "prob",
        ]
    ].head(3)
)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

| Original / Unaudited Claim | Audited Claim (Safe & Defensible) | Methodology Rationale |
| :--- | :--- | :--- |
| *"The Random Forest model predicts which pages will recover lost traffic when refreshed."* | *"The Random Forest model ranks decaying pages by historical search demand and drop velocity to guide human editorial review."* | We do not observe causal post-refresh uplift. The model scores past decay patterns as a decision-support heuristic, not future intervention success. |
| *"Our scoring formula proves Google demoted these specific URLs due to stale content."* | *"We observed a statistical correlation between content age and traffic decline within the analyzed catalog."* | Search engine ranking movements depend on competitive landscape shifts, SERP layout changes, and core algorithm updates that cannot be proven causally from internal traffic logs. |
| *"The ML model achieves 94% Precision in detecting content recovery opportunities."* | *"The baseline heuristic achieved 94.00% Precision@50, while the Random Forest achieved 70.00% Precision@50 and 0.9086 ROC-AUC under a stratified split."* | Accurately distinguishing rule-based baseline precision from machine learning model precision prevents misleading stakeholder expectations about autonomous model accuracy. |
| *"Pages losing traffic have ~50% lower engagement, proving low engagement causes ranking decay."* | *"Observed engagement rate in the dataset is heavily zero-inflated (median 0.00% across declining and growing pages), indicating tracking limits rather than a verified causal driver."* | An honest audit of the raw data shows zero-inflated engagement metrics across cohorts, making strong causal claims regarding engagement unsupported by the data slice. |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.